# Data Profiling

Perfilado de las 18 tablas CSV en `data/raw/`. Objetivo: revisar nulos, duplicados, tipos de datos y estadísticas.

Los datos se leen directamente del CSV original tal como llegaron.


## Configuración

In [21]:
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.2f}".format)

EMAIL_RE = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")

PK_MAP = {
    "university/semesters": "semester_id",
    "university/professors": "professor_id",
    "university/students": "student_id",
    "university/courses": "course_id",
    "university/enrollments": "enrollment_id",
    "university/grades": "grade_id",
    "billing/customers": "customer_id",
    "billing/products": "product_id",
    "billing/subscriptions": "subscription_id",
    "billing/invoices": "invoice_id",
    "billing/invoice_items": "invoice_item_id",
    "billing/payments": "payment_id",
    "crm/accounts": "account_id",
    "crm/contacts": "contact_id",
    "crm/leads": "lead_id",
    "crm/opportunities": "opportunity_id",
    "crm/activities": "activity_id",
}

PROJECT_ROOT = Path("/home/jovyan/work")
RAW_PATH = PROJECT_ROOT / "data" / "raw"

with open(PROJECT_ROOT / "manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

DOMAINS = ["university", "billing", "crm"]


def load_raw(domain, table):
    path = RAW_PATH / domain / f"{table}.csv"
    return pd.read_csv(path)


def list_tables():
    result = []
    for domain in DOMAINS:
        for table in manifest["domains"][domain]:
            result.append((domain, table))
    return result


tables = list_tables()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Tablas a analizar:", len(tables))

PROJECT_ROOT: /home/jovyan/work
Tablas a analizar: 18


## 1. Validación del paquete CSV

Verificacion de que los CSV tengan la misma cantidad de filas que se indica en `manifest.json`.

In [23]:
checks = []

for domain, table in tables:
    df = load_raw(domain, table)
    esperado = manifest["domains"][domain][table]["rows"]
    actual = len(df)
    checks.append({
        "dominio": domain,
        "tabla": table,
        "filas_csv": actual,
        "filas_manifest": esperado,
        "ok": actual == esperado,
    })

check_df = pd.DataFrame(checks)
print("Todas coinciden:", check_df["ok"].all())
check_df

Todas coinciden: True


,dominio,tabla,filas_csv,filas_manifest,ok
0,university,semesters,8,8,True
1,university,professors,200,200,True
2,university,students,5000,5000,True
3,university,courses,300,300,True
4,university,enrollments,25000,25000,True
5,university,grades,60000,60000,True
6,billing,customers,10000,10000,True
7,billing,products,200,200,True
8,billing,subscriptions,15000,15000,True
9,billing,invoices,50000,50000,True


## 2. Perfil de tipos y completitud

Para cada columna registro: ejemplo de dato, tipo de pandas, cantidad de nulos, porcentaje de nulos y valores únicos.

In [31]:
perfil_columnas = []

for domain, table in tables:
    df = load_raw(domain, table)
    columnas = manifest["domains"][domain][table]["cols"]
    total_filas = len(df)

    for col in columnas:
        serie = df[col]
        n_nulos = serie.isna().sum()
        pct_nulos = round(n_nulos / total_filas * 100, 2) if total_filas > 0 else 0
        n_unicos = serie.nunique()

        perfil_columnas.append({
            "dominio": domain,
            "tabla": table,
            "columna": col,
            "example_data": df[col][0],
            "dtype": str(serie.dtype),
            "filas": total_filas,
            "nulos": n_nulos,
            "pct_nulos": pct_nulos,
            "pct_completo": round(100 - pct_nulos, 2),
            "unicos": n_unicos,
        })

schema_df = pd.DataFrame(perfil_columnas)
print("Total columnas perfiladas:", len(schema_df))
schema_df

Total columnas perfiladas: 114


,dominio,tabla,columna,example_data,dtype,filas,nulos,pct_nulos,pct_completo,unicos
0,university,semesters,semester_id,SEM-001,object,8,0,0.00,100.00,8
1,university,semesters,code,2022-1,object,8,0,0.00,100.00,8
2,university,semesters,year,2022,int64,8,0,0.00,100.00,4
3,university,semesters,half,1,int64,8,0,0.00,100.00,2
4,university,semesters,start_date,2022-03-01,object,8,0,0.00,100.00,8
5,university,semesters,end_date,2022-07-15,object,8,0,0.00,100.00,8
6,university,professors,professor_id,PRF-00001,object,200,0,0.00,100.00,200
7,university,professors,first_name,Matias,object,200,0,0.00,100.00,50
8,university,professors,last_name,Sandoval,object,200,0,0.00,100.00,49
9,university,professors,email,matias.sandoval4330@lake.local,object,200,0,0.00,100.00,200


DESCUBRIMIENTO:  
Se encontró 3 columnas con valores nulos:  
billing --> customers --> external_ref : 5000 valores nulos (50%)  
crm --> activities --> contact_id : 5976 valores nulos (29.88%)  
crm --> activities --> opportunity_id : 9985 valores nulos (49.92%)  

  
DESCUBRIMIENTO:  
Todas las columnas con tipo de dato fecha tienen 'date' o 'at' en su nombre  
  
  
DESCUBRIMIENTO:  
Las tablas que tienen emails como columnas son:  
  university --> professors  
  university --> students  
  billing --> customers  
  crm --> contacts  
  crm --> leads  


## 3. Columnas con tipos sospechosos

Columnas de fecha, columnas con valores constantes y columnas de email.

**Busqueda de columnas de tipo "fecha" segun su nombre**

In [32]:
fechas_como_texto = []

for row in schema_df.itertuples():
    nombre = row.columna.lower()
    parece_fecha = ("date" in nombre) or nombre.endswith("_at")
    if parece_fecha and row.dtype == "object":
        fechas_como_texto.append({
            "dominio": row.dominio,
            "tabla": row.tabla,
            "columna": row.columna,
            "dtype": row.dtype,
        })

fechas_texto_df = pd.DataFrame(fechas_como_texto)
print("Columnas de fecha leídas como object:", len(fechas_texto_df))
fechas_texto_df

Columnas de fecha leídas como object: 19


,dominio,tabla,columna,dtype
0,university,semesters,start_date,object
1,university,semesters,end_date,object
2,university,professors,hired_at,object
3,university,students,birth_date,object
4,university,students,enrolled_at,object
5,university,enrollments,enrolled_at,object
6,university,grades,graded_at,object
7,billing,customers,created_at,object
8,billing,subscriptions,start_date,object
9,billing,subscriptions,end_date,object


**Busqueda de columnas con valores constantes**

In [35]:
cols_constantes = schema_df[schema_df["unicos"] == 1][["dominio", "tabla", "columna", "unicos"]]
print("Columnas constantes:", len(cols_constantes))

Columnas constantes: 0


**Analisis de columnas de email**

In [44]:
tablas_con_email = [
    ("university", "professors"),
    ("university", "students"),
    ("billing", "customers"),
    ("crm", "contacts"),
    ("crm", "leads"),
]

longitud_email = []
for domain, table in tablas_con_email:
    df = load_raw(domain, table)
    largos = df["email"].dropna().astype(str).str.len()
    longitud_email.append({
        "tabla": f"{domain}/{table}",
        "min_len": largos.min(),
        "max_len": largos.max(),
        "prom_len": round(largos.mean(), 1),
    })

pd.DataFrame(longitud_email)

,tabla,min_len,max_len,prom_len
0,university/professors,21,38,28.90
1,university/students,20,39,29.10
2,billing/customers,19,39,29.10
3,crm/contacts,19,40,29.10
4,crm/leads,20,38,29.00


## 4. Duplicados

Revisión de llaves primarias, filas repetidas y emails duplicados.

**Busqueda de llaves primarias duplicadas**

In [36]:
dup_pk = []

for tabla_key, pk in PK_MAP.items():
    domain, table = tabla_key.split("/")
    df = load_raw(domain, table)
    dup_pk.append({
        "tabla": tabla_key,
        "pk": pk,
        "filas": len(df),
        "unicos": df[pk].nunique(),
        "duplicados": df[pk].duplicated().sum(),
    })

dup_pk_df = pd.DataFrame(dup_pk)
dup_pk_df

,tabla,pk,filas,unicos,duplicados
0,university/semesters,semester_id,8,8,0
1,university/professors,professor_id,200,200,0
2,university/students,student_id,5000,5000,0
3,university/courses,course_id,300,300,0
4,university/enrollments,enrollment_id,25000,25000,0
5,university/grades,grade_id,60000,60000,0
6,billing/customers,customer_id,10000,10000,0
7,billing/products,product_id,200,200,0
8,billing/subscriptions,subscription_id,15000,15000,0
9,billing/invoices,invoice_id,50000,50000,0


**Busqueda de filas duplicadas**

In [42]:
dup_filas = []
for domain, table in tables:
    df = load_raw(domain, table)
    dup_filas.append({
        "tabla": f"{domain}/{table}",
        "filas": len(df),
        "filas_duplicadas": df.duplicated().sum(),
    })

dup_filas_df = pd.DataFrame(dup_filas)
dup_filas_df[dup_filas_df["filas_duplicadas"] > 0]

,tabla,filas,filas_duplicadas


In [ ]:
**Busqueda de emails duplicadas**

In [45]:
email_check = []

for domain, table in tablas_con_email:
    df = load_raw(domain, table)
    emails = df["email"]

    vacios = (emails.isna() | (emails.astype(str).str.strip() == "")).sum()
    invalidos = 0
    for mail in emails.dropna():
        if not EMAIL_RE.match(str(mail).strip()):
            invalidos += 1

    email_check.append({
        "tabla": f"{domain}/{table}",
        "vacios": vacios,
        "invalidos": invalidos,
        "duplicados": emails.duplicated().sum(),
    })

email_check_df = pd.DataFrame(email_check)
email_check_df

,tabla,vacios,invalidos,duplicados
0,university/professors,0,0,0
1,university/students,0,0,0
2,billing/customers,0,0,0
3,crm/contacts,0,0,2
4,crm/leads,0,0,0


DESCUBRIMIENTO:  
Se encontró dos emails duplicados en crm/contacts

In [47]:
contacts = load_raw("crm", "contacts")
emails_rep = contacts[contacts["email"].duplicated(keep=False)]
print("Registros con email duplicado en contacts:", len(emails_rep))
emails_rep[["contact_id", "email", "account_id"]].sort_values("email")

Registros con email duplicado en contacts: 4


,contact_id,email,account_id
2466,CON-0002467,ignacio.tapia5946@mail.test,ACC-0002715
8391,CON-0008392,ignacio.tapia5946@mail.test,ACC-0004626
3103,CON-0003104,macarena.ortiz4996@lake.local,ACC-0003066
4934,CON-0004935,macarena.ortiz4996@lake.local,ACC-0001325


## 5. Estadísticas descriptivas

Resumen numérico y conteo de categorías en columnas clave.

In [ ]:
Analisis estadistico de valores numericos

In [18]:
stats_numericas = []

for domain, table in tables:
    df = load_raw(domain, table)
    cols = manifest["domains"][domain][table]["cols"]

    for col in cols:
        if pd.api.types.is_numeric_dtype(df[col]):
            stats_numericas.append({
                "dominio": domain,
                "tabla": table,
                "columna": col,
                "count": df[col].count(),
                "mean": round(df[col].mean(), 2),
                "std": round(df[col].std(), 2),
                "min": df[col].min(),
                "max": df[col].max(),
            })

stats_num_df = pd.DataFrame(stats_numericas)
stats_num_df

,dominio,tabla,columna,count,mean,std,min,max
0,university,semesters,year,8,2023.50,1.20,2022,2025
1,university,semesters,half,8,1.50,0.53,1,2
2,university,courses,credits,300,3.98,1.44,2,6
3,university,grades,score,60000,74.88,11.82,24.53,100.00
4,university,grades,weight,60000,0.30,0.12,0.10,0.50
5,billing,products,monthly_price,200,46.25,49.20,5.00,491.47
6,billing,products,active,200,0.85,0.36,False,True
7,billing,invoices,total,50000,135.84,155.62,5.00,4348.24
8,billing,invoice_items,quantity,150000,5.50,2.87,1,10
9,billing,invoice_items,unit_price,150000,42.30,33.69,1.61,697.88


**Analisis de columnas categoricas**  
Las columnas seleccionadas se tomaron en base a un analisis visual de la seccion (2.), tomando aquellas columnas que tienen un valor (unico) bajo comparado con el numero de filas

In [65]:
columnas_a_revisar = [
    ("university", "enrollments", "status"),
    ("billing", "invoices", "status"),
    ("billing", "customers", "segment"),
    ("billing", "customers", "country"),
    ("crm", "leads", "status"),
    ("crm", "opportunities", "stage"),
    ("crm", "activities", "type"),
]

for domain, table, col in columnas_a_revisar:
    df = load_raw(domain, table)
    print(f"\n{domain} --> {table} --> {col}")
    display(df[col].value_counts().to_frame())


university --> enrollments --> status


,count
status,
completed,14931
active,5035
failed,2531
dropped,2503



billing --> invoices --> status


,count
status,
paid,34966
pending,10048
overdue,4986



billing --> customers --> segment


...

,count
status,
new,594
contacted,525
qualified,395
lost,281
converted,205



crm --> opportunities --> stage


,count
stage,
prospect,621
qualification,611
proposal,569
won,476
negotiation,420
lost,303



crm --> activities --> type


,count
type,
email,6904
call,6093
meeting,3016
note,2028
demo,1959
